# LangChain Deep Technical Blog – HuggingFace Version

This notebook uses local HuggingFace models via Transformers + LangChain wrappers to avoid OpenAI billing/quota issues.

In [1]:
# Install Dependencies
!pip install -U langchain langchain-community langchain-huggingface transformers accelerate faiss-cpu sentence-transformers


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.7/112.7 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 44.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 56.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 570.8/570.8 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.8/169.8 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.8 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
 

## Import Libraries

In [2]:
from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate


In [3]:
import os
from getpass import getpass

os.environ["HF_TOKEN"] = getpass("Enter Hugging Face Token: ")

Enter Hugging Face Token: ··········


## Load Local HuggingFace Model

In [4]:
from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline

hf_pipeline = pipeline(
    "text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    token=os.environ["HF_TOKEN"],
    max_new_tokens=100,
    temperature=0.3
)

model = HuggingFacePipeline(pipeline=hf_pipeline)

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'temperature', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


## Basic LLM Call

In [5]:
response = model.invoke(
    "Question: Explain Prompt Engineering in simple terms.\nAnswer:"
)

print(response)

Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: Explain Prompt Engineering in simple terms.
Answer: Prompt Engineering is a process of creating a product or service that meets the needs of the market in a timely and cost-effective manner. It involves identifying the needs of the market, developing a prototype, testing the prototype, refining the product or service, and launching it into the market. The process is iterative and continuous, with continuous feedback and improvement.


## Prompt Template

In [6]:
prompt = PromptTemplate.from_template(
    "Explain {topic} in simple terms."
)

formatted = prompt.invoke({"topic": "Vector Databases"})
print(formatted)


text='Explain Vector Databases in simple terms.'


## Simple Chain

In [7]:
chain = prompt | model

response = chain.invoke({"topic": "Prompt Engineering"})
print(response)


Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Explain Prompt Engineering in simple terms.


## Memory Simulation

In [8]:
chat_history = []

chat_history.append(("User", "Hi"))
chat_history.append(("AI", "Hello!"))
chat_history.append(("User", "What is LangChain?"))

print(chat_history)


[('User', 'Hi'), ('AI', 'Hello!'), ('User', 'What is LangChain?')]


## Custom Tool

In [9]:
def calculator_tool(expression: str):
    return eval(expression)

print(calculator_tool("25 * 8"))


200


## Embeddings

In [10]:
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer("all-MiniLM-L6-v2")
embedding = embed_model.encode("What is LangChain?")
print(len(embedding))


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

384


## FAISS Vector Store

In [11]:
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

texts = [
    "LangChain is a framework for LLM applications.",
    "FAISS is used for vector similarity search."
]

vectorstore = FAISS.from_texts(texts, embedding_model)

results = vectorstore.similarity_search("What is LangChain?")
print(results[0].page_content)


/tmp/ipykernel_2018/3679782293.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


LangChain is a framework for LLM applications.


## Mini RAG Example

In [12]:
context = results[0].page_content

rag_prompt = PromptTemplate.from_template(
    "Answer the question based on context.\nContext: {context}\nQuestion: {question}"
)

rag_chain = rag_prompt | model

answer = rag_chain.invoke({
    "context": context,
    "question": "Explain LangChain"
})

print(answer)


Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Answer the question based on context.
Context: LangChain is a framework for LLM applications.
Question: Explain LangChain's approach to language modeling.


## Use Cases

In [13]:
use_cases = [
    "Customer Support Chatbot",
    "PDF Question Answering",
    "Research Assistant"
]

for case in use_cases:
    print("-", case)


- Customer Support Chatbot
- PDF Question Answering
- Research Assistant


## Conclusion
This notebook demonstrates LangChain concepts using HuggingFace local/free models, avoiding OpenAI quota issues.